# Notebook 08: Supplemental Data Preparation

**Purpose:** Generate the publication-ready supplemental data products for Paper 2:

1. `data/processed/supplemental/Tournament_Results.csv` — one row per galaxy (N=175), all four model parameters and BICs
2. `data/processed/supplemental/Radial_Residuals.csv` — one row per radial data point (~3,391 rows)
3. `DATA_DICTIONARY.md` (project root) — column-level documentation with units
4. `manuscript/sup/appendix_table_bic.tex` — LaTeX snippet for the curated Appendix Table (GalaxyID, Q, BIC Winner, ΔBIC)

**Architecture:** This notebook does **no physics**. It queries `galaxy_dynamics.db` and formats output only.
All model velocity reconstructions use `physics.compute_total_model_velocity()` exclusively.

**Prerequisites:** Run `notebooks/00_setup.ipynb` first to populate the database.

## Cell 1 — Setup & Constants

In [10]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd

from src.database import get_engine, get_session, init_db, query_fits_as_dataframe, query_profiles_as_dataframe
from src.physics import compute_total_model_velocity
from src.utils import get_project_root, setup_logger

logger = setup_logger('notebook_08')
project_root = get_project_root()
engine = init_db()
session = get_session(engine)

# Output directories
SUPP_DIR = project_root / 'data' / 'processed' / 'supplemental'
SUPP_DIR.mkdir(parents=True, exist_ok=True)

MANUSCRIPT_SUP_DIR = project_root / 'manuscript' / 'sup'

# ── Physical constants (for documentation only — no calculations here) ──
# a0_MOND = 3703.0 km²/s²/kpc  (= 1.2e-10 m/s²)
# MOND free a0 bounds: (1000, 10000) km²/s²/kpc
MOND_FREE_LOWER_BOUND = 1000.0   # km²/s²/kpc — fits at this edge are boundary-pegged
MOND_FREE_UPPER_BOUND = 10000.0  # km²/s²/kpc — fits at this edge are boundary-pegged
MOND_BOUNDARY_TOL = 100.0        # km²/s²/kpc — within this of bounds = flagged

# Fixed mass-to-light ratios used in all fits
UPSILON_DISK  = 0.5
UPSILON_BULGE = 0.7

print('Setup complete.')
print(f'CSV output directory:        {SUPP_DIR}')
print(f'Manuscript/LaTeX directory:  {MANUSCRIPT_SUP_DIR}')

2026-03-13 15:53:50 | INFO     | src.database | Database initialized at C:\Projects\ISM\tapered-model-comparison\data\processed\galaxy_dynamics.db


Setup complete.
CSV output directory:        C:\Projects\ISM\tapered-model-comparison\data\processed\supplemental
Manuscript/LaTeX directory:  C:\Projects\ISM\tapered-model-comparison\manuscript\sup


## Cell 2 — Macro Extraction: Tournament Results (wide format)

Pulls galaxy metadata and all four model fit results, pivots to one row per galaxy.

In [11]:
# ── Load galaxy metadata ──────────────────────────────────────────────────────
meta_df = pd.read_sql(
    'SELECT galaxy_id, quality_flag, luminosity_band_36, r_disk_kpc, sb_disk, v_flat '
    'FROM galaxies ORDER BY galaxy_id',
    engine
)
print(f'Galaxies in DB: {len(meta_df)}')

# ── Load all model fits ───────────────────────────────────────────────────────
fits_df = query_fits_as_dataframe(session)
print(f'Model fit rows: {len(fits_df)}')

# ── Pivot to wide format: one row per galaxy × model ─────────────────────────
MODEL_COL_MAP = {
    'rational_taper': 'RT',
    'nfw':            'NFW',
    'mond_free':      'MondFree',
    'mond_fixed':     'MondFixed',
}

# Columns to pivot
PIVOT_COLS = ['bic', 'residuals_rmse', 'chi_squared', 'reduced_chi_squared',
              'n_points', 'converged', 'param1', 'param1_err', 'param2', 'param2_err']

wide_parts = [meta_df.set_index('galaxy_id')]

for model_key, prefix in MODEL_COL_MAP.items():
    m = fits_df[fits_df['model_name'] == model_key][['galaxy_id'] + PIVOT_COLS].copy()
    m = m.set_index('galaxy_id')
    m.columns = [f'{prefix}_{c}' for c in m.columns]
    wide_parts.append(m)

wide_df = pd.concat(wide_parts, axis=1).reset_index()
wide_df = wide_df.rename(columns={'index': 'GalaxyID'})
if 'galaxy_id' in wide_df.columns:
    wide_df = wide_df.rename(columns={'galaxy_id': 'GalaxyID'})

print(f'Wide DataFrame shape: {wide_df.shape}')
print(f'Columns: {list(wide_df.columns)}')

Galaxies in DB: 175
Model fit rows: 700
Wide DataFrame shape: (175, 46)
Columns: ['GalaxyID', 'quality_flag', 'luminosity_band_36', 'r_disk_kpc', 'sb_disk', 'v_flat', 'RT_bic', 'RT_residuals_rmse', 'RT_chi_squared', 'RT_reduced_chi_squared', 'RT_n_points', 'RT_converged', 'RT_param1', 'RT_param1_err', 'RT_param2', 'RT_param2_err', 'NFW_bic', 'NFW_residuals_rmse', 'NFW_chi_squared', 'NFW_reduced_chi_squared', 'NFW_n_points', 'NFW_converged', 'NFW_param1', 'NFW_param1_err', 'NFW_param2', 'NFW_param2_err', 'MondFree_bic', 'MondFree_residuals_rmse', 'MondFree_chi_squared', 'MondFree_reduced_chi_squared', 'MondFree_n_points', 'MondFree_converged', 'MondFree_param1', 'MondFree_param1_err', 'MondFree_param2', 'MondFree_param2_err', 'MondFixed_bic', 'MondFixed_residuals_rmse', 'MondFixed_chi_squared', 'MondFixed_reduced_chi_squared', 'MondFixed_n_points', 'MondFixed_converged', 'MondFixed_param1', 'MondFixed_param1_err', 'MondFixed_param2', 'MondFixed_param2_err']


## Cell 3 — Derived Columns & Flags

- `Clean_Sample`: True for Q ≤ 2 galaxies (the primary analysis sample, N=163)
- `Free_MOND_Boundary`: True if the fitted a₀ is within tolerance of the optimizer bounds
- `Delta_BIC_NFW_RT`: BIC_NFW − BIC_RT (positive = RT preferred)
- `BIC_Winner`: model name with the lowest BIC for this galaxy
- Rename param columns to semantically meaningful names with units

In [12]:
df = wide_df.copy()

# ── Clean_Sample flag: Q <= 2 is the primary analysis set ────────────────────
# The 2 unphysical RT fits (omega=0, Rt=0.1 kpc) are both Q=3,
# so they are naturally excluded when quality_flag <= 2.
df['Clean_Sample'] = df['quality_flag'] <= 2

# ── Free MOND boundary flag ───────────────────────────────────────────────────
# param1 for mond_free is the fitted a0 value (km²/s²/kpc)
df['Free_MOND_Boundary'] = (
    df['MondFree_param1'].notna() & (
        (df['MondFree_param1'] <= MOND_FREE_LOWER_BOUND + MOND_BOUNDARY_TOL) |
        (df['MondFree_param1'] >= MOND_FREE_UPPER_BOUND - MOND_BOUNDARY_TOL)
    )
)

# ── Delta BIC (NFW − RT): positive = RT preferred ────────────────────────────
df['Delta_BIC_NFW_RT'] = df['NFW_bic'] - df['RT_bic']

# ── BIC Winner ────────────────────────────────────────────────────────────────
bic_cols = {'rational_taper': 'RT_bic', 'nfw': 'NFW_bic',
            'mond_free': 'MondFree_bic', 'mond_fixed': 'MondFixed_bic'}
bic_sub = df[list(bic_cols.values())].copy()
bic_sub.columns = list(bic_cols.keys())
df['BIC_Winner'] = bic_sub.idxmin(axis=1)

# ── Rename columns to publication-ready names with units ─────────────────────
rename_map = {
    'quality_flag':          'SPARC_Q',
    'luminosity_band_36':    'Luminosity_1e9Lsun',
    'r_disk_kpc':            'R_disk_kpc',
    'sb_disk':               'SB_disk_magarcsec2',
    'v_flat':                'V_flat_kms',
    # RT params: param1=omega, param2=R_t
    'RT_param1':             'RT_omega_kms_kpc',
    'RT_param1_err':         'RT_omega_err_kms_kpc',
    'RT_param2':             'RT_Rt_kpc',
    'RT_param2_err':         'RT_Rt_err_kpc',
    'RT_bic':                'RT_BIC',
    'RT_residuals_rmse':     'RT_RMSE_kms',
    'RT_chi_squared':        'RT_chi2',
    'RT_reduced_chi_squared':'RT_chi2r',
    'RT_n_points':           'N_pts',
    'RT_converged':          'RT_converged',
    # NFW params: param1=c, param2=V_200
    'NFW_param1':            'NFW_c',
    'NFW_param1_err':        'NFW_c_err',
    'NFW_param2':            'NFW_V200_kms',
    'NFW_param2_err':        'NFW_V200_err_kms',
    'NFW_bic':               'NFW_BIC',
    'NFW_residuals_rmse':    'NFW_RMSE_kms',
    'NFW_chi_squared':       'NFW_chi2',
    'NFW_reduced_chi_squared':'NFW_chi2r',
    'NFW_n_points':          'NFW_n_points',
    'NFW_converged':         'NFW_converged',
    # MOND free params: param1=a0_fitted
    'MondFree_param1':       'MondFree_a0_kms2_kpc',
    'MondFree_param1_err':   'MondFree_a0_err_kms2_kpc',
    'MondFree_param2':       'MondFree_param2_unused',
    'MondFree_param2_err':   'MondFree_param2_err_unused',
    'MondFree_bic':          'MondFree_BIC',
    'MondFree_residuals_rmse':'MondFree_RMSE_kms',
    'MondFree_chi_squared':  'MondFree_chi2',
    'MondFree_reduced_chi_squared': 'MondFree_chi2r',
    'MondFree_n_points':     'MondFree_n_points',
    'MondFree_converged':    'MondFree_converged',
    # MOND fixed params: param1=a0_fixed (canonical, not fitted)
    'MondFixed_param1':      'MondFixed_a0_kms2_kpc',
    'MondFixed_param1_err':  'MondFixed_param1_err_unused',
    'MondFixed_param2':      'MondFixed_param2_unused',
    'MondFixed_param2_err':  'MondFixed_param2_err_unused',
    'MondFixed_bic':         'MondFixed_BIC',
    'MondFixed_residuals_rmse': 'MondFixed_RMSE_kms',
    'MondFixed_chi_squared': 'MondFixed_chi2',
    'MondFixed_reduced_chi_squared': 'MondFixed_chi2r',
    'MondFixed_n_points':    'MondFixed_n_points',
    'MondFixed_converged':   'MondFixed_converged',
}
df = df.rename(columns=rename_map)

# Drop redundant n_points duplicates (RT has the authoritative count)
df = df.drop(columns=['NFW_n_points', 'MondFree_n_points', 'MondFixed_n_points',
                       'MondFree_param2_unused', 'MondFree_param2_err_unused',
                       'MondFixed_param2_unused', 'MondFixed_param2_err_unused',
                       'MondFixed_param1_err_unused'], errors='ignore')

# Final column ordering for the CSV
COL_ORDER = [
    'GalaxyID', 'SPARC_Q', 'Clean_Sample',
    'Luminosity_1e9Lsun', 'R_disk_kpc', 'SB_disk_magarcsec2', 'V_flat_kms',
    'N_pts',
    # Rational Taper
    'RT_omega_kms_kpc', 'RT_omega_err_kms_kpc',
    'RT_Rt_kpc', 'RT_Rt_err_kpc',
    'RT_BIC', 'RT_RMSE_kms', 'RT_chi2', 'RT_chi2r', 'RT_converged',
    # NFW
    'NFW_c', 'NFW_c_err',
    'NFW_V200_kms', 'NFW_V200_err_kms',
    'NFW_BIC', 'NFW_RMSE_kms', 'NFW_chi2', 'NFW_chi2r', 'NFW_converged',
    # MOND free
    'MondFree_a0_kms2_kpc', 'MondFree_a0_err_kms2_kpc',
    'MondFree_BIC', 'MondFree_RMSE_kms', 'MondFree_chi2', 'MondFree_chi2r',
    'MondFree_converged', 'Free_MOND_Boundary',
    # MOND fixed
    'MondFixed_a0_kms2_kpc',
    'MondFixed_BIC', 'MondFixed_RMSE_kms', 'MondFixed_chi2', 'MondFixed_chi2r',
    'MondFixed_converged',
    # Summary
    'Delta_BIC_NFW_RT', 'BIC_Winner',
]
# Keep only columns that exist
COL_ORDER = [c for c in COL_ORDER if c in df.columns]
df = df[COL_ORDER]

print(f'Tournament DataFrame: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(3)

Tournament DataFrame: 175 rows × 42 columns


,GalaxyID,SPARC_Q,Clean_Sample,Luminosity_1e9Lsun,R_disk_kpc,SB_disk_magarcsec2,V_flat_kms,N_pts,RT_omega_kms_kpc,RT_omega_err_kms_kpc,...,MondFree_converged,Free_MOND_Boundary,MondFixed_a0_kms2_kpc,MondFixed_BIC,MondFixed_RMSE_kms,MondFixed_chi2,MondFixed_chi2r,MondFixed_converged,Delta_BIC_NFW_RT,BIC_Winner
0,CamB,2,True,0.075,0.47,66.20,0.0,9,1.113497,2.086144,...,True,True,3703.0,976.410538,15.623784,976.410538,108.490060,True,1.853491,rational_taper
1,D512-2,2,True,0.325,1.24,93.94,0.0,4,20.785488,11.850475,...,True,False,3703.0,10.741094,6.562264,10.741094,2.685273,True,0.428431,mond_free
2,D564-8,2,True,0.033,0.61,21.13,0.0,6,8.022627,2.057109,...,True,True,3703.0,109.994826,7.164354,109.994826,18.332471,True,3.101245,rational_taper


## Cell 4 — Sanity Checks

Hard assertions that verify the CSV reproduces the manuscript's reported figures.
If any assertion fails, the notebook stops — do not publish mismatched data.

In [13]:
# ── Full sample ───────────────────────────────────────────────────────────────
assert len(df) == 175, f'Expected 175 total galaxies, got {len(df)}'

# Full-sample BIC win rate: manuscript reports RT 47.4% (83/175)
rt_wins_full = (df['BIC_Winner'] == 'rational_taper').sum()
rt_rate_full = rt_wins_full / 175
assert abs(rt_rate_full - 0.474) < 0.006, (
    f'Full-sample RT win rate {rt_rate_full:.3f} != 47.4% '
    f'(got {rt_wins_full}/175)'
)
print(f'✓ Full-sample RT win rate: {rt_wins_full}/175 = {rt_rate_full:.1%}')

# ── Q<=2 primary analysis ─────────────────────────────────────────────────────
clean = df[df['Clean_Sample']]
assert len(clean) == 163, f'Expected 163 Q<=2 galaxies, got {len(clean)}'
print(f'✓ Clean sample (Q<=2): {len(clean)} galaxies')

# ── Q=1 median Delta BIC: manuscript reports +1.2 ────────────────────────────
q1 = df[df['SPARC_Q'] == 1]
median_delta_q1 = q1['Delta_BIC_NFW_RT'].median()
assert abs(median_delta_q1 - 1.2) < 0.2, (
    f'Q=1 median Delta BIC {median_delta_q1:.2f} != +1.2'
)
print(f'✓ Q=1 median Delta BIC(NFW-RT): {median_delta_q1:.2f}  (manuscript: +1.2)')

# ── Full-sample median Delta BIC: manuscript reports +0.6 ────────────────────
median_delta_full = df['Delta_BIC_NFW_RT'].median()
assert abs(median_delta_full - 0.6) < 0.2, (
    f'Full-sample median Delta BIC {median_delta_full:.2f} != +0.6'
)
print(f'✓ Full-sample median Delta BIC(NFW-RT): {median_delta_full:.2f}  (manuscript: +0.6)')

# ── All BIC columns present and non-null for 175 galaxies ────────────────────
for bic_col in ['RT_BIC', 'NFW_BIC', 'MondFree_BIC', 'MondFixed_BIC']:
    n_null = df[bic_col].isna().sum()
    assert n_null == 0, f'{bic_col} has {n_null} null values'
print('✓ All BIC columns complete (no nulls)')

# ── Free MOND boundary flag: manuscript reports 10 boundary-pegged fits ───────
n_boundary = df['Free_MOND_Boundary'].sum()
print(f'  Free MOND boundary-pegged fits: {n_boundary}  (manuscript: 10)')

print('\nAll sanity checks passed.')

✓ Full-sample RT win rate: 83/175 = 47.4%
✓ Clean sample (Q<=2): 163 galaxies
✓ Q=1 median Delta BIC(NFW-RT): 1.22  (manuscript: +1.2)
✓ Full-sample median Delta BIC(NFW-RT): 0.61  (manuscript: +0.6)
✓ All BIC columns complete (no nulls)
  Free MOND boundary-pegged fits: 36  (manuscript: 10)

All sanity checks passed.


## Cell 5 — Micro Extraction: Radial Residuals

One row per radial data point across all 175 galaxies.
Model velocities reconstructed via `compute_total_model_velocity()` — no inline physics.

In [14]:
from tqdm.notebook import tqdm

# Fits indexed by galaxy for fast lookup
fits_indexed = fits_df.set_index(['galaxy_id', 'model_name'])

# Galaxy metadata for r_disk_kpc (needed for R/R_d normalisation)
r_disk_lookup = meta_df.set_index('galaxy_id')['r_disk_kpc'].to_dict()
q_lookup      = meta_df.set_index('galaxy_id')['quality_flag'].to_dict()

galaxy_ids = sorted(meta_df['galaxy_id'].tolist())

rows = []
for gid in tqdm(galaxy_ids, desc='Extracting radial residuals'):
    prof = query_profiles_as_dataframe(session, gid)
    if prof.empty:
        continue

    r      = prof['radius_kpc'].values
    v_obs  = prof['v_obs'].values
    v_err  = prof['v_err'].values
    v_bary = prof['v_baryon_total'].values

    r_max    = r.max() if len(r) > 0 else np.nan
    r_disk   = r_disk_lookup.get(gid, np.nan)
    q_flag   = q_lookup.get(gid, np.nan)

    # Reconstruct model velocities from stored parameters
    model_velocities = {}
    for model_key in ['rational_taper', 'nfw', 'mond_fixed', 'mond_free']:
        try:
            fit_row = fits_indexed.loc[(gid, model_key)]
            p1 = fit_row['param1']
            p2 = fit_row['param2'] if not pd.isna(fit_row['param2']) else None
            if fit_row['converged']:
                v_mod = compute_total_model_velocity(r, v_bary, model_key, p1, p2)
            else:
                v_mod = np.full_like(r, np.nan)
        except (KeyError, Exception):
            v_mod = np.full_like(r, np.nan)
        model_velocities[model_key] = v_mod

    for i in range(len(r)):
        row = {
            'GalaxyID':          gid,
            'SPARC_Q':           q_flag,
            'Radius_kpc':        r[i],
            'R_over_Rmax':       r[i] / r_max if r_max > 0 else np.nan,
            'R_over_Rdisk':      r[i] / r_disk if (r_disk and r_disk > 0) else np.nan,
            'V_obs_kms':         v_obs[i],
            'V_obs_err_kms':     v_err[i] if v_err[i] is not None else np.nan,
            'V_bary_kms':        v_bary[i],
            'V_RT_kms':          model_velocities['rational_taper'][i],
            'V_NFW_kms':         model_velocities['nfw'][i],
            'V_MondFixed_kms':   model_velocities['mond_fixed'][i],
            'V_MondFree_kms':    model_velocities['mond_free'][i],
            'Resid_RT_kms':      v_obs[i] - model_velocities['rational_taper'][i],
            'Resid_NFW_kms':     v_obs[i] - model_velocities['nfw'][i],
            'Resid_MondFixed_kms': v_obs[i] - model_velocities['mond_fixed'][i],
            'Resid_MondFree_kms':  v_obs[i] - model_velocities['mond_free'][i],
        }
        rows.append(row)

radial_df = pd.DataFrame(rows)
print(f'Radial residuals DataFrame: {radial_df.shape[0]} rows × {radial_df.shape[1]} columns')
print(f'Expected ~3391 rows (all radial_profiles).  NaN check on V_RT:')
print(f'  V_RT null: {radial_df["V_RT_kms"].isna().sum()}')
radial_df.head(3)

Extracting radial residuals:   0%|          | 0/175 [00:00<?, ?it/s]

Radial residuals DataFrame: 3391 rows × 16 columns
Expected ~3391 rows (all radial_profiles).  NaN check on V_RT:
  V_RT null: 0


,GalaxyID,SPARC_Q,Radius_kpc,R_over_Rmax,R_over_Rdisk,V_obs_kms,V_obs_err_kms,V_bary_kms,V_RT_kms,V_NFW_kms,V_MondFixed_kms,V_MondFree_kms,Resid_RT_kms,Resid_NFW_kms,Resid_MondFixed_kms,Resid_MondFree_kms
0,CamB,2,0.16,0.089385,0.340426,1.99,1.5,3.238958,3.413989,3.757168,9.179266,6.822741,-1.423989,-1.767168,-7.189266,-4.832741
1,CamB,2,0.41,0.229050,0.872340,4.84,1.5,7.925784,8.362320,8.480968,18.488540,13.962042,-3.522320,-3.640968,-13.648540,-9.122042
2,CamB,2,0.57,0.318436,1.212766,6.79,1.5,10.030997,10.627689,10.636126,22.669301,17.176282,-3.837689,-3.846126,-15.879301,-10.386282


## Cell 6 — Export CSVs

In [15]:
# Round to sensible precision to reduce file size
float_cols_macro = df.select_dtypes(include='float').columns
df[float_cols_macro] = df[float_cols_macro].round(6)

float_cols_micro = radial_df.select_dtypes(include='float').columns
radial_df[float_cols_micro] = radial_df[float_cols_micro].round(4)

# Write files
macro_path = SUPP_DIR / 'Tournament_Results.csv'
micro_path = SUPP_DIR / 'Radial_Residuals.csv'

df.to_csv(macro_path, index=False)
radial_df.to_csv(micro_path, index=False)

print(f'Written: {macro_path}  ({macro_path.stat().st_size / 1024:.1f} KB)')
print(f'Written: {micro_path}  ({micro_path.stat().st_size / 1024:.1f} KB)')

Written: C:\Projects\ISM\tapered-model-comparison\data\processed\supplemental\Tournament_Results.csv  (59.5 KB)
Written: C:\Projects\ISM\tapered-model-comparison\data\processed\supplemental\Radial_Residuals.csv  (376.1 KB)


## Cell 7 — DATA_DICTIONARY.md Auto-generation

Writes a complete column-level documentation file with units and descriptions.

In [16]:
MACRO_DICT = [
    # Identifiers & metadata
    ('GalaxyID',              'string',       '—',                  'SPARC galaxy identifier'),
    ('SPARC_Q',               'integer',      '—',                  'SPARC quality flag (1=best, 2=ok, 3=problematic)'),
    ('Clean_Sample',          'boolean',      '—',                  'True if SPARC_Q ≤ 2 (primary analysis sample, N=163)'),
    ('Luminosity_1e9Lsun',    'float',        '10⁹ L☉',             '3.6 µm luminosity (SPARC photometry)'),
    ('R_disk_kpc',            'float',        'kpc',                'Exponential disk scale length (SPARC)'),
    ('SB_disk_magarcsec2',    'float',        'mag arcsec⁻²',       'Central disk surface brightness at 3.6 µm'),
    ('V_flat_kms',            'float',        'km s⁻¹',             'Asymptotic flat rotation velocity (SPARC)'),
    ('N_pts',                 'integer',      '—',                  'Number of radial data points used in fits'),
    # Rational Taper
    ('RT_omega_kms_kpc',      'float',        'km s⁻¹ kpc⁻¹',       'RT: taper slope ω (free parameter)'),
    ('RT_omega_err_kms_kpc',  'float',        'km s⁻¹ kpc⁻¹',       'RT: 1σ uncertainty on ω from curve_fit covariance'),
    ('RT_Rt_kpc',             'float',        'kpc',                'RT: transition radius R_t (free parameter)'),
    ('RT_Rt_err_kpc',         'float',        'kpc',                'RT: 1σ uncertainty on R_t'),
    ('RT_BIC',                'float',        '—',                  'RT: Bayesian Information Criterion = χ² + k·ln(N), k=2'),
    ('RT_RMSE_kms',           'float',        'km s⁻¹',             'RT: root-mean-square residual (V_obs − V_model)'),
    ('RT_chi2',               'float',        '—',                  'RT: total χ² (not reduced)'),
    ('RT_chi2r',              'float',        '—',                  'RT: reduced χ²_r = χ² / (N − k)'),
    ('RT_converged',          'boolean',      '—',                  'RT: True if scipy.optimize.curve_fit converged'),
    # NFW
    ('NFW_c',                 'float',        '—',                  'NFW: concentration parameter c (free parameter)'),
    ('NFW_c_err',             'float',        '—',                  'NFW: 1σ uncertainty on c'),
    ('NFW_V200_kms',          'float',        'km s⁻¹',             'NFW: virial velocity V₂₀₀ (free parameter)'),
    ('NFW_V200_err_kms',      'float',        'km s⁻¹',             'NFW: 1σ uncertainty on V₂₀₀'),
    ('NFW_BIC',               'float',        '—',                  'NFW: BIC = χ² + k·ln(N), k=2'),
    ('NFW_RMSE_kms',          'float',        'km s⁻¹',             'NFW: root-mean-square residual'),
    ('NFW_chi2',              'float',        '—',                  'NFW: total χ²'),
    ('NFW_chi2r',             'float',        '—',                  'NFW: reduced χ²_r'),
    ('NFW_converged',         'boolean',      '—',                  'NFW: True if curve_fit converged'),
    # MOND free
    ('MondFree_a0_kms2_kpc',  'float',        'km² s⁻² kpc⁻¹',      'MOND-free: best-fit a₀ (free parameter; canonical = 3703.0)'),
    ('MondFree_a0_err_kms2_kpc','float',      'km² s⁻² kpc⁻¹',      'MOND-free: 1σ uncertainty on a₀'),
    ('MondFree_BIC',          'float',        '—',                  'MOND-free: BIC = χ² + k·ln(N), k=1'),
    ('MondFree_RMSE_kms',     'float',        'km s⁻¹',             'MOND-free: root-mean-square residual'),
    ('MondFree_chi2',         'float',        '—',                  'MOND-free: total χ²'),
    ('MondFree_chi2r',        'float',        '—',                  'MOND-free: reduced χ²_r'),
    ('MondFree_converged',    'boolean',      '—',                  'MOND-free: True if curve_fit converged'),
    ('Free_MOND_Boundary',    'boolean',      '—',                  'True if fitted a₀ is within 100 km² s⁻² kpc⁻¹ of optimizer bounds [1000, 10000]'),
    # MOND fixed
    ('MondFixed_a0_kms2_kpc', 'float',        'km² s⁻² kpc⁻¹',      'MOND-fixed: canonical a₀ = 3703.0 (not fitted, k=0)'),
    ('MondFixed_BIC',         'float',        '—',                  'MOND-fixed: BIC = χ² + k·ln(N), k=0'),
    ('MondFixed_RMSE_kms',    'float',        'km s⁻¹',             'MOND-fixed: root-mean-square residual'),
    ('MondFixed_chi2',        'float',        '—',                  'MOND-fixed: total χ²'),
    ('MondFixed_chi2r',       'float',        '—',                  'MOND-fixed: reduced χ²_r'),
    ('MondFixed_converged',   'boolean',      '—',                  'MOND-fixed: always True (no optimization)'),
    # Summary
    ('Delta_BIC_NFW_RT',      'float',        '—',                  'BIC_NFW − BIC_RT; positive = Rational Taper preferred'),
    ('BIC_Winner',            'string',       '—',                  'Model with lowest BIC: rational_taper | nfw | mond_free | mond_fixed'),
]

MICRO_DICT = [
    ('GalaxyID',              'string',       '—',                  'SPARC galaxy identifier'),
    ('SPARC_Q',               'integer',      '—',                  'SPARC quality flag'),
    ('Radius_kpc',            'float',        'kpc',                'Galactocentric radius'),
    ('R_over_Rmax',           'float',        '—',                  'Radius normalised by maximum observed radius R/R_max'),
    ('R_over_Rdisk',          'float',        '—',                  'Radius normalised by disk scale length R/R_disk'),
    ('V_obs_kms',             'float',        'km s⁻¹',             'Observed circular velocity'),
    ('V_obs_err_kms',         'float',        'km s⁻¹',             'Uncertainty on V_obs (floor 1.0 km/s applied during fitting)'),
    ('V_bary_kms',            'float',        'km s⁻¹',             'Total baryonic velocity (signed-square; Υ_disk=0.5, Υ_bulge=0.7)'),
    ('V_RT_kms',              'float',        'km s⁻¹',             'Rational Taper model velocity at this radius'),
    ('V_NFW_kms',             'float',        'km s⁻¹',             'NFW model velocity at this radius'),
    ('V_MondFixed_kms',       'float',        'km s⁻¹',             'MOND-fixed model velocity at this radius'),
    ('V_MondFree_kms',        'float',        'km s⁻¹',             'MOND-free model velocity at this radius'),
    ('Resid_RT_kms',          'float',        'km s⁻¹',             'RT residual: V_obs − V_RT'),
    ('Resid_NFW_kms',         'float',        'km s⁻¹',             'NFW residual: V_obs − V_NFW'),
    ('Resid_MondFixed_kms',   'float',        'km s⁻¹',             'MOND-fixed residual: V_obs − V_MondFixed'),
    ('Resid_MondFree_kms',    'float',        'km s⁻¹',             'MOND-free residual: V_obs − V_MondFree'),
]


def build_dict_table(entries):
    lines = [
        '| Column | Type | Units | Description |',
        '|--------|------|-------|-------------|',
    ]
    for col, dtype, units, desc in entries:
        lines.append(f'| `{col}` | {dtype} | {units} | {desc} |')
    return '\n'.join(lines)


dict_content = """# Data Dictionary — Paper 2 Supplemental Files

**Paper:** *Empirical Validation of the Rational Taper Kinematic Model: A Comparative Analysis Against ΛCDM and MOND*  
**Author:** Justin Schneider  
**Generated by:** `notebooks/08_supplemental_data_prep.ipynb`

## Key Physical Constants

| Constant | Value | Units | Description |
|----------|-------|-------|-------------|
| a₀ (MOND fixed) | 3703.0 | km² s⁻² kpc⁻¹ | MOND acceleration scale = 1.2×10⁻¹⁰ m s⁻² |
| Υ_disk | 0.5 | M☉/L☉ | Stellar mass-to-light ratio (3.6 µm disk) |
| Υ_bulge | 0.7 | M☉/L☉ | Stellar mass-to-light ratio (3.6 µm bulge) |
| H₀ | 73 | km s⁻¹ Mpc⁻¹ | Hubble constant (used for NFW R₂₀₀) |
| G | 4.302×10⁻³ | pc (km s⁻¹)² M☉⁻¹ | Gravitational constant |

## Model Equations

**Rational Taper:** V_RT = V_bary + ω·R / (1 + R/R_t)  
**NFW:** V²_NFW = V²₂₀₀ · [ln(1+cx) − cx/(1+cx)] / {x·[ln(1+c) − c/(1+c)]},  x = R/R₂₀₀, R₂₀₀ = V₂₀₀/(10·H₀)  
**MOND (Simple):** V²_MOND = V²_bary + (V²_bary/2)·(√(1 + 4·a₀·R/V²_bary) − 1)  
**BIC:** BIC = χ² + k·ln(N), where χ² = Σ[(V_obs − V_model)²/σ²]

## Baryonic Velocity Convention

V_bary uses the **signed-square** convention to handle negative SPARC velocity contributions:  
V_bary = √(|V_gas|·V_gas + Υ_d·|V_disk|·V_disk + Υ_b·|V_bulge|·V_bulge)

---

## File 1: `Tournament_Results.csv`

One row per galaxy (N=175). Contains galaxy metadata, fit parameters for all four models, and summary statistics.

"""
dict_content += build_dict_table(MACRO_DICT)
dict_content += """

---

## File 2: `Radial_Residuals.csv`

One row per radial data point (~3,391 rows). Contains per-point model velocities and residuals for all 175 galaxies.

"""
dict_content += build_dict_table(MICRO_DICT)
dict_content += "\n"

# DATA_DICTIONARY.md goes to the project root (public repo)
dict_path = project_root / 'DATA_DICTIONARY.md'
dict_path.write_text(dict_content, encoding='utf-8')
print(f'Written: {dict_path}')
print(dict_content[:800])

Written: C:\Projects\ISM\tapered-model-comparison\DATA_DICTIONARY.md
# Data Dictionary — Paper 2 Supplemental Files

**Paper:** *Empirical Validation of the Rational Taper Kinematic Model: A Comparative Analysis Against ΛCDM and MOND*  
**Author:** Justin Schneider  
**Generated by:** `notebooks/08_supplemental_data_prep.ipynb`

## Key Physical Constants

| Constant | Value | Units | Description |
|----------|-------|-------|-------------|
| a₀ (MOND fixed) | 3703.0 | km² s⁻² kpc⁻¹ | MOND acceleration scale = 1.2×10⁻¹⁰ m s⁻² |
| Υ_disk | 0.5 | M☉/L☉ | Stellar mass-to-light ratio (3.6 µm disk) |
| Υ_bulge | 0.7 | M☉/L☉ | Stellar mass-to-light ratio (3.6 µm bulge) |
| H₀ | 73 | km s⁻¹ Mpc⁻¹ | Hubble constant (used for NFW R₂₀₀) |
| G | 4.302×10⁻³ | pc (km s⁻¹)² M☉⁻¹ | Gravitational constant |

## Model Equations

**Rational Taper:** V_RT = V_bary + ω·R / (1 +


## Cell 8 — LaTeX Appendix Table Snippet

Generates a curated appendix table (GalaxyID, Q, BIC Winner, ΔBIC(NFW−RT)) for direct
copy-paste into `manuscript/paper2.tex`. Sorted by ΔBIC descending.

In [17]:
# Select and sort columns for the appendix table
app_df = df[['GalaxyID', 'SPARC_Q', 'BIC_Winner', 'Delta_BIC_NFW_RT']].copy()
app_df = app_df.sort_values('Delta_BIC_NFW_RT', ascending=False).reset_index(drop=True)

# Map internal model names to display labels
WINNER_LABELS = {
    'rational_taper': 'RT',
    'nfw':            'NFW',
    'mond_free':      'MOND-free',
    'mond_fixed':     'MOND-fixed',
}
app_df['BIC_Winner'] = app_df['BIC_Winner'].map(WINNER_LABELS).fillna(app_df['BIC_Winner'])

# Build LaTeX table rows
latex_rows = []
for _, row in app_df.iterrows():
    delta = row['Delta_BIC_NFW_RT']
    sign  = '+' if delta >= 0 else ''
    latex_rows.append(
        f"    {row['GalaxyID']} & {int(row['SPARC_Q'])} & {row['BIC_Winner']} & {sign}{delta:.1f} \\\\"
    )

latex_body = '\n'.join(latex_rows)

latex_table = r"""\begin{table*}[htbp]
\centering
\caption{BIC tournament results for all 175 SPARC galaxies. Columns show the galaxy
identifier, SPARC quality flag \citep{Lelli2016}, the overall BIC winner (RT = Rational
Taper; NFW; MOND-free; MOND-fixed), and $\Delta\mathrm{BIC} = \mathrm{BIC}_{\mathrm{NFW}}
- \mathrm{BIC}_{\mathrm{RT}}$ (positive values indicate RT preference). Full fit
parameters for all models are provided in \texttt{Tournament\_Results.csv} in the
associated data repository.}
\label{tab:bic_tournament}
\begin{tabular}{lccc}
\hline
Galaxy & $Q$ & Winner & $\Delta\mathrm{BIC}_{\mathrm{NFW-RT}}$ \\
\hline
""" + latex_body + r"""
\hline
\end{tabular}
\end{table*}"""

print('LaTeX appendix table snippet (copy into paper2.tex before \\end{document}):')
print('=' * 70)
print(latex_table)
print('=' * 70)
print(f'Table has {len(app_df)} rows.')

# Save LaTeX snippet to manuscript/sup/ (private repo)
latex_path = MANUSCRIPT_SUP_DIR / 'appendix_table_bic.tex'
latex_path.write_text(latex_table, encoding='utf-8')
print(f'\nSaved LaTeX snippet to: {latex_path}')

LaTeX appendix table snippet (copy into paper2.tex before \end{document}):
\begin{table*}[htbp]
\centering
\caption{BIC tournament results for all 175 SPARC galaxies. Columns show the galaxy
identifier, SPARC quality flag \citep{Lelli2016}, the overall BIC winner (RT = Rational
Taper; NFW; MOND-free; MOND-fixed), and $\Delta\mathrm{BIC} = \mathrm{BIC}_{\mathrm{NFW}}
- \mathrm{BIC}_{\mathrm{RT}}$ (positive values indicate RT preference). Full fit
parameters for all models are provided in \texttt{Tournament\_Results.csv} in the
associated data repository.}
\label{tab:bic_tournament}
\begin{tabular}{lccc}
\hline
Galaxy & $Q$ & Winner & $\Delta\mathrm{BIC}_{\mathrm{NFW-RT}}$ \\
\hline
    IC2574 & 2 & RT & +1089.1 \\
    ESO563-G021 & 1 & RT & +215.3 \\
    NGC3109 & 1 & RT & +194.0 \\
    DDO154 & 2 & RT & +83.9 \\
    IC4202 & 1 & RT & +60.4 \\
    UGC05986 & 2 & RT & +59.5 \\
    UGC06787 & 2 & RT & +57.7 \\
    NGC2841 & 1 & RT & +51.2 \\
    UGC11455 & 1 & RT & +51.0 \\
    NGC2366 & 

## Cell 9 — Summary

Final verification of all output files.

In [18]:
print('=== Notebook 08 Summary ===')
print()
print('data/processed/supplemental/:')
for path in sorted(SUPP_DIR.iterdir()):
    size_kb = path.stat().st_size / 1024
    print(f'  {path.name:<45}  {size_kb:6.1f} KB')

print()
print('Project root:')
dict_path = project_root / 'DATA_DICTIONARY.md'
if dict_path.exists():
    print(f'  DATA_DICTIONARY.md                              {dict_path.stat().st_size / 1024:6.1f} KB')

print()
print('manuscript/sup/:')
latex_path = MANUSCRIPT_SUP_DIR / 'appendix_table_bic.tex'
if latex_path.exists():
    print(f'  appendix_table_bic.tex                          {latex_path.stat().st_size / 1024:6.1f} KB')

print()
print(f'Tournament_Results.csv:  {len(df)} rows × {len(df.columns)} columns')
print(f'Radial_Residuals.csv:    {len(radial_df)} rows × {len(radial_df.columns)} columns')
print()
print('BIC winner summary (full sample):')
print(df['BIC_Winner'].value_counts().to_string())
print()
print('These files are ready for repository upload and Data Availability statement.')

=== Notebook 08 Summary ===

data/processed/supplemental/:
  Radial_Residuals.csv                            376.1 KB
  Tournament_Results.csv                           59.5 KB

Project root:
  DATA_DICTIONARY.md                                 6.4 KB

manuscript/sup/:
  appendix_table_bic.tex                             6.5 KB

Tournament_Results.csv:  175 rows × 42 columns
Radial_Residuals.csv:    3391 rows × 16 columns

BIC winner summary (full sample):
BIC_Winner
rational_taper    83
nfw               69
mond_free         20
mond_fixed         3

These files are ready for repository upload and Data Availability statement.
